*   The concise implementation uses PyTorch objects for the model, loss, optimizer, and data loader.
*   The goal is to map each shortcut back to the manual training loop.

In [5]:
import math
import random
import numpy as np
import torch

# 3.5.1 Defining the Model

## 1. Intuition

*   A concise model uses a framework layer instead of manually storing `w` and `b`.
*   `torch.nn.Linear` is a PyTorch layer that computes a linear transformation: input times weights plus bias.

## Common types of `torch.nn` modules

| Category                  | Common modules                                                                                                                                                    | Purpose                                         |
| ------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------- |
| **Dense layers**          | `Linear`, `Bilinear`, `LazyLinear`                                                                                                                                | Fully connected transformations                 |
| **Convolutional**         | `Conv1d`, `Conv2d`, `Conv3d`, `ConvTranspose1d/2d/3d`, `LazyConv2d`                                                                                               | CNN feature extraction                          |
| **Pooling**               | `MaxPool1d/2d/3d`, `AvgPool1d/2d/3d`, `AdaptiveAvgPool2d`, `AdaptiveMaxPool2d`, `FractionalMaxPool2d`, `LPPool2d`                                                 | Downsampling                                    |
| **Normalization**         | `BatchNorm1d/2d/3d`, `LayerNorm`, `GroupNorm`, `InstanceNorm1d/2d/3d`, `LocalResponseNorm`, `RMSNorm`                                                             | Stabilize training                              |
| **Activation**            | `ReLU`, `LeakyReLU`, `PReLU`, `ELU`, `GELU`, `SELU`, `SiLU`, `Mish`, `Sigmoid`, `Tanh`, `Softmax`, `Softplus`, `Softsign`, `Hardtanh`, `Hardswish`, `Hardsigmoid` | Nonlinearities                                  |
| **Dropout**               | `Dropout`, `Dropout1d`, `Dropout2d`, `Dropout3d`, `AlphaDropout`                                                                                                  | Regularization                                  |
| **Recurrent**             | `RNN`, `LSTM`, `GRU`, `RNNCell`, `LSTMCell`, `GRUCell`                                                                                                            | Sequence modeling                               |
| **Transformer**           | `MultiheadAttention`, `Transformer`, `TransformerEncoder`, `TransformerDecoder`, `TransformerEncoderLayer`, `TransformerDecoderLayer`                             | Attention-based models                          |
| **Embedding**             | `Embedding`, `EmbeddingBag`                                                                                                                                       | Learn vector representations of discrete tokens |
| **Padding**               | `ZeroPad2d`, `ReflectionPad2d`, `ReplicationPad2d`, `ConstantPad2d`, etc.                                                                                         | Add padding to tensors                          |
| **Upsampling**            | `Upsample`, `UpsamplingNearest2d`, `UpsamplingBilinear2d`                                                                                                         | Increase spatial resolution                     |
| **Pixel operations**      | `PixelShuffle`, `PixelUnshuffle`                                                                                                                                  | Super-resolution and rearrangement              |
| **Containers**            | `Sequential`, `ModuleList`, `ModuleDict`, `ParameterList`, `ParameterDict`                                                                                        | Organize models                                 |
| **Distance / Similarity** | `CosineSimilarity`, `PairwiseDistance`                                                                                                                            | Compute similarity metrics                      |
| **Loss functions**        | `CrossEntropyLoss`, `MSELoss`, `L1Loss`, `BCEWithLogitsLoss`, `NLLLoss`, `HuberLoss`, `SmoothL1Loss`, `TripletMarginLoss`, etc.                                   | Training objectives                             |


## 2. Why this exists

*   Framework layers reduce boilerplate and automatically register parameters so optimizers can find them.

## 3. Examples

*   Create a one-output linear layer for two input features.

In [6]:
net = torch.nn.Linear(2, 1) # Creates a linear layer with 2 inputs and 1 output features (e.g. y = x*w + b); net.weight and net.bias are randomly initialized and will be learned during training
X = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

# Equivalent to X @ net.weight.T + net.bias; net() here is the net we defined earlier, and it's a callable instance (like a function) because Linear in PyTorch is designed to be callable
# Computes 1 * w1 + 2 * w2 + b first, then 3 * w1 + 4 * w2 + b second; w and b are net.weight and net.bias randomly generated by torch.nn.Linear
y_hat = net(X)

y_hat.shape

torch.Size([2, 1])

*   Inspect the parameter shapes.
*   Even though the values of `net.weight` and `net.bias` are random, the shapes are determined by layer architecture.
> *   Weight shape is `[# of output, # of input]`, the opposite of the shape passed to `torch.nn.Linear`
> *   Bias shape is `[# of output]`

In [7]:
weight_shape = net.weight.shape
bias_shape = net.bias.shape
weight_shape, bias_shape

(torch.Size([1, 2]), torch.Size([1]))

## 4. Step-by-step breakdown

* `torch.nn.Linear(2, 1)` expects 2 input features and produces 1 output per example.

* `net(X)` runs the layer's forward computation.

* The output has shape `(2, 1)` because there are 2 examples and 1 output per example.

* `net.weight` and `net.bias` are learnable parameters registered inside the layer.

## 5. Connection to ML systems

*   This replaces the manual `linreg(X, w, b)` function from chapter 3.4 and separate parameter tensors.

## 6. Common confusion points

- A layer initializes parameters (`net.weight` and `net.bias`) before training.
- Registered parameters can be found by `net.parameters()`.
- Output shape `(batch_size, 1)` may differ from label shape `(batch_size,)`.
> - Make sure labels `y` shape matches the model output `y_hat` shape (or is compatible with the loss function).
- Concise does not mean conceptually different, shorter code can hide the same operations.

In [8]:
net.parameters()

<generator object Module.parameters at 0x7a9c46ec6dc0>

# 3.5.2 Defining the Loss Function

## 1. Intuition

*   A loss module is a framework object that computes loss.
*   Mean squared error loss measures average squared difference between predictions and labels.

## 2. Why this exists

*   Using a built-in loss avoids rewriting common formulas and gives consistent reduction options.

## 3. Examples

*   Use PyTorch's mean squared error loss.
*   `torch.nn.MSELoss()` (Mean Square Loss)computes the mean squared error between predictions and target values:

    $$
    \text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(y_i^{pred} - y_i^{true})^2
    $$

    Where:
    - $y_i^{pred}$ is the predicted value
    - $y_i^{true}$ is the true target value
    - $N$ is the number of values being compared

    The calculation:
    1. Compute the difference between prediction and target.
    2. Square each difference.
    3. Take the average of the squared differences.

In [9]:
loss_fn = torch.nn.MSELoss()
y_hat = torch.tensor([[1.0], [3.0]])
y = torch.tensor([[2.0], [1.0]])
loss = loss_fn(y_hat, y) # Calculates the average or (prediction - actual)^^ 2, or ((1-2)^^2 + (3-1)^^2) / 2
loss

tensor(2.5000)

## 4. Step-by-step breakdown

*   `torch.nn.MSELoss()` creates a loss object.

*   Calling `loss_fn(y_hat, y)` compares predictions and labels.

*   By default, this returns the mean squared error as one scalar.

*   The prediction and label shapes are both `(2, 1)`, so no accidental broadcasting is needed.



## 5. Connection to ML systems

*   This replaces the manual squared-loss function and explicit `.mean()` reduction.

## 6. Common confusion points

- Built-in losses still need correctly shaped inputs (prediction and actual shape must match).
- MSE means mean squared error.
- A loss object can hide reduction details, so check defaults (like how MSE averages the loss with square roots).
> * For example, by calling `help(torch.nn.MSELoss)`, you can see that MSE's __init__ defined `str` as `mean`:

    ```
    Methods inherited from _Loss:
      __init__(self, size_average=None, reduce=None, reduction: str = 'mean') -> None
    ```

- The loss is just a tensor computed from predictions and labels, and it stores the computation graph needed for backpropagation.

# 3.5.3 Defining the Optimization Algorithm

## 1. Intuition

*   A PyTorch optimizer stores references to parameters and updates them using gradients.

*   SGD is stochastic gradient descent, the same update idea used in the from-scratch version.



## 2. Why this exists

*   The optimizer object handles repeated parameter updates and gradient-clearing patterns more reliably than handwritten code.

## 3. Examples

*   Create an SGD optimizer for the model parameters.

In [10]:
net = torch.nn.Linear(2,1) # Creates a linear layer with 2 input features and 1 output feature
trainer = torch.optim.SGD(net.parameters(), lr=0.03) # Creates an SGD optimizer with net's learnable params (weights & bias) and learn rate of 0.03
list(net.parameters())[0].shape # Use list() to convert net.parameters(), which is a generator, to a list, and inspect the shape of list()[0] or the shape of weights, list()[1] would be bias

torch.Size([1, 2])

## 4. Step-by-step breakdown

*   `net.parameters()` returns the trainable parameters stored in the layer.

*   `torch.optim.SGD(...)` creates an optimizer that will update those parameters.

*   `lr=0.03` sets the learning rate.

*   The optimizer does not update anything until `.step()` is called after gradients exist.

## 5. Connection to ML systems

*   This replaces the manual `sgd([w, b], lr, batch_size)` function.

## 6. Common confusion points

*   Creating an optimizer does not train the model.
*   The optimizer needs parameters, not predictions.
*   `.step()` updates parameters using the computed gradients (e.g. `loss`).
*   `.zero_grad()` clears old gradients before the next backward pass.

# 3.5.4 Training

## 1. Intuition

*   Concise training still follows the same order: batch, prediction, loss, backward, update.

*   The difference is that PyTorch objects perform some details for us.



## 2. Why this exists

*   This version is closer to real PyTorch code while still small enough to trace line by line.

## 3. Examples

*   Train a linear layer on synthetic data.

In [13]:
torch.manual_seed(0)
X = torch.randn(20, 2) # Creates a tensor of shape (20, 2), or 20 examples with 2 features/example
y = (X @ torch.tensor([2.0, -3.4]) + 4.2).reshape(-1,1) # Change the shape of y from (20, ) to (20, 1) to match X, else the loss computation might broadcast

loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X, y),
    batch_size=5, # Each batch contains 5 samples (20 samples -> 4 batches)
    shuffle=True)

net = torch.nn.Linear(2, 1) # Creates a linear layer with 2 input features and 1 output feature
loss_fn = torch.nn.MSELoss() # We will use MSELoss to calculate the loss of this model
trainer = torch.optim.SGD(net.parameters(), lr=0.03) # We will use SGD at a learn rate of 0.03 to train this model

for Xb, yb in loader:
  trainer.zero_grad(); loss_fn(net(Xb), yb).backward(); trainer.step()
  # 1 Clear out existing cached grad (loss
  # 2 Compute the loss on the current batch (5 examples)
  # 3 Update the params (weights & bias) using SGD

loss_fn(net(X), y)
# Compute the MSE loss on the entire dataset after training.
# Since the data is perfectly linear, the loss should approach 0 after sufficient training.
# With only 1 epoch (4 SGD updates), the loss usually won't be extremely close to zero

tensor(20.6533, grad_fn=<MseLossBackward0>)

*   If you' like to see `loss_fn(net(X), y)` getting to nearly 0, add more epoch (training loop) to the model:

    ```
    for epoch in range(100):
        for Xb, yb in loader:
            trainer.zero_grad()

            pred = net(Xb)
            loss = loss_fn(pred, yb)

            loss.backward()
            trainer.step()

    print(loss_fn(net(X), y))
    ```
*   Other methods include making your batch size bigger (e.g. the entire dataset in one batch), or force your model to train until loss minimizes as much as possible, like `if loss_fn(net(X), y) < 1e-8`.
*   Training for more epochs increases training time, and on real-world datasets excessive training can lead to overfitting.
*   However, in this toy example the data is perfectly linear and noise-free, so a linear model can fit it almost exactly without overfitting.
*   In larger neural networks with many layers and parameters, it's important to train for a reasonable number of epochs. Training too long can cause the model to memorize the training data and overfit, so it's common to monitor validation loss and stop training once it stops improving.

## 4. Step-by-step breakdown

*   The synthetic labels follow a known linear rule.
*   The DataLoader returns minibatches.
*   `trainer.zero_grad()` clears gradients from the previous batch.
*   `loss_fn(net(Xb), yb)` computes the scalar batch loss (which, at the end, we asked it for full dataset loss of the entire model).
*   `.backward()` computes gradients.
*   `trainer.step()` updates the layer parameters.



## 5. Connection to ML systems

*   This is the practical PyTorch version of the from-scratch loop. The conceptual order is unchanged.

## 6. Common confusion points

*   Semicolons keep this tiny example compact, but production code should usually use separate lines for readability.
*   One pass over the loader (the entirety of 20 examples) is one epoch here.
*   Built-in optimizers still require explicit gradient clearing.
*   A final loss after one epoch may still be high.



# 3.5.5 Summary

## 1. Intuition

*   The concise implementation replaces manual pieces with PyTorch objects.
> *   Model: `torch.nn.Linear`.
> *   Loss: `torch.nn.MSELoss`.
> *   Optimizer: `torch.optim.SGD`.
> *   Data: `DataLoader`.



## 2. Why this exists

*   These objects reduce boilerplate once the manual loop is understood.

## 3. Examples

*   Map concise objects to from-scratch pieces.

In [ ]:
mapping = {
    "torch.nn.Linear": "linreg plus parameters",
    "torch.nn.MSELoss": "squared_loss plus reduction",
    "torch.optim.SGD": "sgd update function",
    "DataLoader": "data_iter function",
}

## 4. Step-by-step breakdown

*   Each PyTorch object corresponds to a manual concept.

*   The concise version is shorter because common patterns are packaged.

*   The package does not remove the underlying sequence of computation.



## 5. Connection to ML systems

*   This mapping is the main skill needed to read higher-level training code without losing the execution flow.

## 6. Common confusion points

- Framework objects are wrappers around concepts you already saw manually.
- Shorter code can be harder to debug if you cannot map it back.
- Always know which object owns parameters.
- Always know when gradients are computed and cleared.

# 3.5.6 Exercises

## 1. Intuition

*   These exercises check whether you can modify concise PyTorch training code safely.

## 2. Why this exists

*   Real work often means changing model dimensions, learning rates, batch sizes, or loss reductions.

## 3. Examples

*   Exercise 1: create a linear model with three input features.

In [14]:
net3 = torch.nn.Linear(3, 1)
X3 = torch.randn(4, 3)
y3_hat = net3(X3)
y3_hat.shape #Should be (4, 1) since X3 has the shape of (4, 3) or 4 examples with 3 features each, and Linear(3,1) maps those 3 input features into 1 output feature per example.

torch.Size([4, 1])

*   Exercise 2: inspect optimizer parameter groups.

In [16]:
opt = torch.optim.SGD(net3.parameters(), lr=0.01)
len(opt.param_groups), opt.param_groups[0]["lr"] # Should have 1 param group (SGD); lr=0.01

(1, 0.01)

## 4. Step-by-step breakdown

*   Exercise 1 checks input-output dimensions.

*   Exercise 2 shows that optimizers store parameter groups and settings.

*   A parameter group is a collection of parameters updated with the same optimizer settings.



## 5. Connection to ML systems

*   Parameter groups become useful when different parts of a model need different learning rates.

## 6. Common confusion points

- The first argument to `Linear` is input feature count.
- The second argument to `Linear` is output count.
- Optimizer settings live in parameter groups.
- Changing a learning rate changes update size, not the model formula.